# 01. Linked lists

**Tier:** Deep. **Why you'd use this:** Values live in nodes that each point to the next. There is no random index. To print the sequence you start at the head and follow `next` until you hit `None`.

**After this lesson you can:** build a chain from a Python list; walk a chain back into a list of values.
About 25 minutes. **Uses:** [Classes](../../course/11-classes/01-classes-lesson.ipynb), [Loops](../../course/03-control-flow/02-loops-lesson.ipynb)


## Quick review

1. What does an object attribute store?
2. How do you stop a `while` loop when a reference becomes missing?

<details><summary>Answers</summary>

Named fields on the instance, such as `val` and `next`. Loop while the reference is not `None`. Skim [Classes](../../course/11-classes/01-classes-lesson.ipynb) if attribute access is new.
</details>


## Recognize it

Data is a chain of nodes, each holding a value and a link to the next node (or `None`). The prompt talks about a "linked list", "head of the list", or "follow next until null".

Signals: no integer index into the middle; the only way forward is the `next` pointer. A look-alike that is not this pattern: a Python `list`, which is array-backed and supports `values[i]` in constant time.


## Why it works this way

A Python list stores references in a contiguous buffer, so index `i` is one jump. A singly linked list stores each value in its own node with a pointer to the successor. Finding the k-th node means k hops from the head. That cost is the trade: splicing a node next to a known neighbor is pointer rewiring without shifting a buffer, but scans and middle access pay linear time.

Building from a Python list needs a head and a tail. The first value becomes both. Each later value becomes a new node hanging off `tail.next`, then `tail` advances. Forgetting to advance `tail` leaves every new node attached to the same place or dropped. Walking out is the dual: start at `head`, append `node.val`, set `node = node.next`, stop at `None`.

Empty input is `None` for the head, and walking `None` yields an empty value list. Mutating `head` inside the walk is fine when you only need the values — the caller's reference to the original head is unchanged if you rebind the local name. Losing the only reference to the head without storing it elsewhere orphans the chain for the garbage collector.

See [Objects](../../internals/01-objects/01-names-lesson.ipynb) for names versus objects: `head` is a name pointing at a node object; rebinding `head` does not rewrite other names that still point at the old node.
Reversing a list rewires three pointers per step: previous, current, and the saved next. Merging two sorted lists advances the smaller head and hangs it off a dummy tail. Cycle detection runs two speeds until they meet or the fast runner hits `None`. All of those algorithms assume you are comfortable creating nodes, losing no references, and stopping cleanly at `None`.

Dummy head nodes are a common idiom when the real head might be replaced: allocate a fake node whose `next` starts as the real head, splice relative to the dummy, then return `dummy.next`. This lesson stays with the plain head/tail build so the pointer story stays visible without that extra object.


## When to use it

Use a linked chain when you will splice or reverse links often and rarely need random access. Prefer a Python list when you need indexes, slicing, or cache-friendly scans. Prefer a deque when you need cheap pushes and pops at both ends of a sequence you still treat as a contiguous collection.


## Brute force, and why it's slow

There is no useful nested alternative for a single walk. The costly mistake is rebuilding with repeated concatenation that copies the growing prefix on every node.


In [1]:
def to_values_slow(head):
    copies = 0
    out = []
    while head is not None:
        out = out + [head.val]  # copies the whole prefix each time
        copies += len(out)
        head = head.next
    return out, copies


class Node:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def chain(values):
    head = None
    for value in reversed(values):
        head = Node(value, head)
    return head


for size in (50, 100, 200):
    print(size, to_values_slow(chain(list(range(size))))[1])


50 1275
100 5050
200 20100


## Key insight

Keep a `tail` while building so each new node links in constant time. While reading, follow `next` and append into a result list — one visit per node.


## Worked example

Build a singly linked list from a Python list of values, then walk it back into a list. Empty input becomes `None` and walks to `[]`.


In [2]:

def to_list(values: list[int]):
    # 1. Remember the first node and the growing tail.
    head = None
    tail = None
    for value in values:
        node = Node(value)
        if head is None:
            head = node
        else:
            tail.next = node
        # 2. Advance the tail every time.
        tail = node
    return head


def to_values(head) -> list[int]:
    # 3. Follow next until None.
    out = []
    while head is not None:
        out.append(head.val)
        head = head.next
    return out


print(to_values(to_list([1, 2, 3])))
print(to_values(to_list([])))
print(to_list([7]).val)


[1, 2, 3]
[]
7


## Trace

`to_list([1, 2, 3])`:

| value | head | tail | links |
|---|---|---|---|
| 1 | node1 | node1 | node1.next is None |
| 2 | node1 | node2 | node1 → node2 |
| 3 | node1 | node3 | node1 → node2 → node3 |

`to_values` then visits 1, 2, 3 and stops at `None`.


## Cost

Building and walking each visit every value once: time grows with `n`. Extra memory is the `n` nodes plus the output list. The concatenate-each-step walk pays about `n²` element copies.


## Variants

Not solved here: [Reverse Linked List](../../challenges/problems/13-reverse-linked-list.ipynb), [Merge Two Sorted Lists](../../challenges/problems/14-merge-two-sorted-lists.ipynb), [Linked List Cycle](../../challenges/problems/24-linked-list-cycle.ipynb).


## Common mistake

This bug is silent. The builder never advances `tail`, so every new node overwrites `head.next` and earlier nodes are dropped. There is no traceback. The print is `[1, 3]`, and the answer is `[1, 2, 3]`.


In [3]:

def to_list_wrong(values):
    head = None
    tail = None
    for value in values:
        node = Node(value)
        if head is None:
            head = node
            tail = node  # noqa: F841 — bug never uses tail
        else:
            head.next = node  # should link from tail, then advance tail
    return head


def to_values(head):
    out = []
    while head is not None:
        out.append(head.val)
        head = head.next
    return out


print(to_values(to_list_wrong([1, 2, 3])))


[1, 3]


The fix links `tail.next = node` and then sets `tail = node`. Run `to_values(to_list([1, 2, 3]))` and you get `[1, 2, 3]`.


In [4]:
print(to_values(to_list([1, 2, 3])))


[1, 2, 3]


## Python idioms

Build backward by prepending: `head = Node(value, head)` inside a reversed loop. That needs no separate tail. The forward form with an explicit tail matches how many splice algorithms talk about the growing end.


In [5]:
def to_list_prepend(values: list[int]):
    head = None
    for value in reversed(values):
        head = Node(value, head)
    return head


print(to_values(to_list_prepend([1, 2, 3])))


[1, 2, 3]


## Recap

- A node holds a value and a `next` link.
- Build with a head and a moving tail.
- Walk with `while head is not None`, then step `head = head.next`.

Next: [practice](01-traversal-practice.ipynb).
